<h2>Setup</h2>

In [2]:
import time

from pyvo.dal import TAPService

In [4]:
# Setup the TAP service using the GCP endpoint; white-listed for access from 'data-int.lsst.cloud'
# svc = TAPService("http://35.192.145.251/tap")
svc = TAPService("http://data-int.lsst.cloud/api/ppdbtap")  # Dan's current endpoint.

In [6]:
# Simple query example
from lsst.rsp import get_tap_service
svc = get_tap_service("ppdbtap")
svc.run_async("SELECT COUNT(*) FROM ppdb.DiaObject").resultstable

<VOTable length=1>
 col_0  
 int64  
--------
11891504

In [7]:
def query(sql):
    """Run a query, return the results as a DataFrame, get basic performance timing, and 
    print along with row count."""
    start = time.perf_counter()
    res = svc.run_async(sql).resultstable
    end = time.perf_counter()
    df = res.to_table().to_pandas()
    print(f"Query took {end - start:.3f} seconds and returned {len(df)} rows")
    return df

<h2>ID</h2>

In [ ]:
query("SELECT * FROM ppdb.DiaObject WHERE diaObjectId=24624704620855420")

<h2>Cone Search</h2>

In [5]:
query("""
SELECT diaObjectId, ra, dec
FROM ppdb.DiaObject
WHERE CONTAINS(
POINT('ICRS', ra, dec),
CIRCLE('ICRS', 186.8, 7.0, 0.1)) = 1
""")

Query took 5.945 seconds and returned 8273 rows


,diaObjectId,ra,dec
0,24624704466714748,186.780702,7.091353
1,24624704466714748,186.780704,7.091352
2,24624704466714748,186.780703,7.091353
3,24624704466714748,186.780705,7.091353
4,24624704620855420,186.843132,7.014296
...,...,...,...
8268,24624703949766814,186.775822,7.053721
8269,24624703949766814,186.775822,7.053719
8270,24624703949766814,186.775822,7.053719
8271,24624721416945740,186.756986,6.946242


<h2>Nearest Neighbor Search</h2>

In [ ]:
# This seems to work for de-duplicating the records based an validityStart.
query("""
SELECT 
    o1.diaObjectId AS id1,
    o2.diaObjectId AS id2,
    DISTANCE(POINT('ICRS', o1.ra, o1.dec), POINT('ICRS', o2.ra, o2.dec)) AS d
FROM ppdb.DiaObject AS o1
JOIN ppdb.DiaObject AS o2
  ON o1.diaObjectId <> o2.diaObjectId
 AND o1.validityStart = o2.validityStart
WHERE CONTAINS(POINT('ICRS', o1.ra, o1.dec),
               CIRCLE('ICRS', 186.84, 7.01, 0.05)) = 1
  AND DISTANCE(POINT('ICRS', o1.ra, o1.dec),
               POINT('ICRS', o2.ra, o2.dec)) < 0.02;
""")

<h2>Join</h2>

In [ ]:
query("""SELECT * FROM ppdb.DiaSource ds 
LEFT JOIN ppdb.DiaObject dob ON dob.diaObjectId = ds.diaObjectId
WHERE dob.diaObjectId=24624704620855420""")

<h2>Table scan</h2>

In [ ]:
query("SELECT * FROM ppdb.DiaObject WHERE r_psfFluxMean BETWEEN 1090.0 and 1100.0")

<h2>Scratch</h2>

In [ ]:
raise RuntimeError("Comment me out to run scratch queries")

In [ ]:
tbl = query("SELECT diaObjectId FROM ppdb.DiaObject as diaobj WHERE diaobj.ra >= 186 and diaobj.ra < 187") 
tbl.head()

In [ ]:
query("SELECT COUNT(DISTINCT diaObjectId) FROM ppdb.DiaObject") 

In [ ]:
query("SELECT MIN(ra) AS ra_min, MAX(ra) AS ra_max, MIN(dec) AS dec_min, MAX(dec) AS dec_max FROM ppdb.DiaObject")

In [ ]:
query("SELECT * FROM ppdb.DiaSource WHERE diaObjectId=24624704620855420")

In [ ]:
query("SELECT * FROM ppdb.DiaForcedSource WHERE diaObjectId=24624704620855420")

In [ ]:
query("""SELECT * 
FROM ppdb.DiaObject 
WHERE CONTAINS(POINT('ICRS', ra, dec), 
CIRCLE('ICRS', 186.8, 7.0, 0.1)) = 1""")

In [ ]:
query("""SELECT * FROM ppdb.DiaSource ds 
LEFT JOIN ppdb.DiaObject dob ON dob.diaObjectId = ds.diaObjectId
WHERE dob.diaObjectId=24624704620855420
""")

In [ ]:
query("query("SELECT * FROM ppdb.DiaSource WHERE diaObjectId=24624704620855420")

In [ ]:
query("SELECT * FROM ppdb.DiaObject 
WHERE r_psfFluxMean BETWEEN 1090.0 and 1100.0")

In [ ]:
nn2_sql = """
SELECT TOP 1000 o1.diaObjectId AS id1, o2.diaObjectId AS id2,
       DISTANCE(POINT('ICRS', o1.ra, o1.dec), POINT('ICRS', o2.ra, o2.dec)) AS d
FROM   ppdb.DiaObject o1
JOIN   ppdb.DiaObject o2
  ON   o1.diaObjectId < o2.diaObjectId
WHERE  CONTAINS(POINT('ICRS', o1.ra, o1.dec),
                CIRCLE('ICRS', 186.5, 7.05, 0.5))=1
 AND   DISTANCE(POINT('ICRS', o1.ra, o1.dec),
                POINT('ICRS', o2.ra, o2.dec)) < 0.5;
"""
query(nn2_sql)

In [8]:
query("SELECT diaObjectId, ra, dec FROM ppdb_lsstcam.DiaObject WHERE CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 186.8, 7.0, 0.1)) = 1")

Query took 3.067 seconds and returned 0 rows


,diaObjectId,ra,dec
